# Chapitre 3 : Exploration et nettoyage des données

*MLSMM2151 Data Science · Séance 2, vendredi 27 novembre, matin · Corentin Vande Kerckhove*

> **CONSEIL** : commencez par enregistrer une copie de ce notebook dans votre Google Drive
> (`Fichier` > `Enregistrer une copie dans Drive`) pour pouvoir prendre vos propres notes.

## Notebook 1 sur 3 : Analyse exploratoire et contrôle qualité

**Durée estimée : 15 minutes.**

### Objectifs

À la fin de ce notebook, vous saurez :

- faire le tour d'un jeu de données inconnu en cinq commandes ;
- repérer une colonne dont le type est faux, et la convertir sans inventer
  d'information ;
- écrire un contrôle de cohérence, compter ses violations, et séparer le
  constat de la décision.


## 3.1. Analyse exploratoire des données (EDA)

Avant de nettoyer, il faut savoir ce qu'on a entre les mains. La routine est
toujours la même, et chaque commande a un angle mort : c'est pour cela qu'on
les utilise toutes.

| Commande | La question | Ce qu'elle ne vous dit pas |
|---|---|---|
| `shape` | Combien de lignes, combien de colonnes ? | Ce qu'est une ligne |
| `info()` | Quel type, combien de valeurs remplies par colonne ? | Si le type est le bon |
| `describe()` | Quel ordre de grandeur pour les colonnes numériques ? | Tout ce qui est stocké en texte |
| `value_counts()` | Quelles modalités, et combien de chacune ? | Les colonnes numériques |
| un histogramme | Quelle forme a la distribution ? | Les valeurs isolées, écrasées par l'échelle |


In [ ]:
# Cette ligne récupère le fichier utils/donnees.py du dépôt du cours.
!curl -fsSL -o donnees.py https://raw.githubusercontent.com/cvandekerckh/mlsmm2151/main/utils/donnees.py

import pandas as pd
import matplotlib.pyplot as plt
from donnees import charger

In [ ]:
annonces = charger("airbnb_raw")

In [ ]:
annonces.shape

In [ ]:
annonces.info()

**Lecture.** 6 721 annonces, 38 colonnes. Deux choses sautent aux yeux :
`license` n'a **aucune** valeur remplie, et seize colonnes sont du texte
(`str`, ou `object` selon votre version de pandas), dont `price` et
`host_since`, qui devraient être un nombre et une date.


In [ ]:
# On écarte les identifiants : leur moyenne n'a aucun sens.
# .T affiche une ligne par colonne, plus lisible quand il y en a beaucoup.
annonces.drop(columns=["id", "host_id"]).describe().T.round(1)

**Lecture.** La colonne `max` d'abord : `minimum_nights` monte à 500 nuits,
`beds` à 36 lits. Des valeurs à vérifier, pas forcément des erreurs. Ensuite
ce qui **manque** : pas de ligne `price`. Le prix est stocké en texte, et
`describe()` l'ignore sans rien dire. C'est son angle mort.


In [ ]:
annonces["room_type"].value_counts()

Quatre modalités, très déséquilibrées : 4 931 logements entiers, 1 757
chambres privées, et seulement 19 chambres partagées et 14 chambres d'hôtel.
Une catégorie de 14 lignes posera un problème à l'encodage, au chapitre 4.


In [ ]:
# Un histogramme se lit par quelqu'un d'autre que vous : un titre, des axes nommés.
fig, ax = plt.subplots(figsize=(7, 3.5))
annonces["review_scores_rating"].plot.hist(bins=40, ax=ax)
ax.set_title("Note globale des annonces Airbnb à Bruxelles")
ax.set_xlabel("note sur 5")
ax.set_ylabel("nombre d'annonces")
plt.show()

**Lecture.** La distribution est écrasée contre 5 : sur Airbnb, 4,5 n'est pas
une bonne note, c'est une note moyenne. Une variable dont presque toutes les
valeurs se ressemblent distingue mal les annonces entre elles. Et
l'histogramme ignore les annonces sans note, sans le signaler.


## 3.2. Contrôle qualité des données

Deux vérifications, dans cet ordre : **le type** (un nombre est-il stocké
comme un nombre ?), puis **la cohérence** (les valeurs ont-elles un sens,
seules et entre elles ?).

### Des types faux


In [ ]:
annonces[["price", "host_response_rate", "host_is_superhost"]].head()

Le prix `$64.00` contient un symbole dollar, et les prix au-delà de 999 une
virgule de milliers : `$1,234.00`. On retire les deux, puis on convertit.
`regex=False` dit à pandas de chercher le caractère `$` tel quel.


In [ ]:
# On crée une nouvelle colonne : la colonne brute reste là pour comparer.
annonces["prix"] = (
    annonces["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)
annonces["prix"].describe()

Le prix est enfin visible : 5 701 annonces ont un prix, la médiane est de 87,
la moyenne de 110, et le maximum de **9 999** par nuit. Gardez ce chiffre, il
revient au notebook 2. Même geste pour le taux de réponse : on retire le `%`
et on divise par 100.


In [ ]:
annonces["taux_reponse"] = annonces["host_response_rate"].str.rstrip("%").astype(float) / 100
annonces["taux_reponse"].describe()

Reste `host_is_superhost`, qui vaut `t` ou `f`. La conversion la plus rapide
est une comparaison. Elle est aussi la plus dangereuse : comparez les deux
résultats.


In [ ]:
# Version naïve : une comparaison. Un manquant n'est pas égal à "t", il devient donc False.
naif = annonces["host_is_superhost"] == "t"
naif.value_counts(dropna=False)

In [ ]:
# Version correcte : un dictionnaire. Ce qui n'est ni "t" ni "f" reste manquant.
annonces["superhost"] = annonces["host_is_superhost"].map({"t": 1, "f": 0})
annonces["superhost"].value_counts(dropna=False)

La comparaison a transformé **165 hôtes dont on ne sait rien** en « pas
Superhost », sans avertissement. `map` les laisse manquants. Une conversion
ne doit jamais inventer d'information.


### Hack-Time 🛠️

Nous venons de convertir `host_response_rate` en proportion. La colonne
`host_acceptance_rate` a exactement le même défaut. À votre tour, dans la
cellule ci-dessous :

1. Créez une colonne `taux_acceptation` à partir de `host_acceptance_rate`,
   en proportion entre 0 et 1. Une piste : reprenez la cellule
   `taux_reponse` et changez le nom de la colonne.
2. Décrivez-la avec `describe()`, puis comptez les hôtes dont le taux vaut
   exactement 0 : `(annonces["taux_acceptation"] == 0).sum()`.
3. Médiane et taux nuls, en une phrase : qu'est-ce qu'un taux d'acceptation de
   0 % peut bien vouloir dire pour une annonce publiée sur Airbnb ?

**Quatre minutes.**

**Défi optionnel**
Comparez la médiane de `taux_acceptation` entre Superhosts et non-Superhosts :
`annonces.groupby("superhost")["taux_acceptation"].median()`. L'écart vous
surprend-il ? Que deviennent les 165 hôtes au statut manquant dans ce calcul ?

In [ ]:
# Votre code ici

### Des contrôles de cohérence

Un contrôle de cohérence est une règle qui **doit** être vraie, écrite en
code, dont on compte les violations. Deux règles sur les dates : un premier
avis ne peut pas précéder l'inscription de l'hôte, et aucun avis ne peut être
postérieur à la date du fichier, le 21 juin 2025. Pour comparer des dates, il
faut d'abord qu'elles en soient : `pd.to_datetime`, que le notebook 3 détaille.


In [ ]:
for colonne in ["host_since", "first_review", "last_review"]:
    annonces[colonne] = pd.to_datetime(annonces[colonne])

In [ ]:
# Règle 1 : le premier avis ne peut pas précéder l'inscription de l'hôte.
avis_avant_hote = annonces["first_review"] < annonces["host_since"]
print(f"{avis_avant_hote.sum()} annonces ont un premier avis antérieur à l'inscription de leur hôte")

In [ ]:
# Ces lignes viennent-elles de nombreux hôtes, ou de quelques-uns ?
annonces.loc[avis_avant_hote, "host_id"].value_counts()

In [ ]:
# Règle 2 : aucun avis ne peut être postérieur à la date du fichier.
DATE_INSTANTANE = pd.Timestamp("2025-06-21")
avis_apres_instantane = annonces["last_review"] > DATE_INSTANTANE
print(f"{avis_apres_instantane.sum()} annonces ont un dernier avis postérieur au {DATE_INSTANTANE.date()}")
print(f"Le plus tardif date du {annonces['last_review'].max().date()}")

Un fichier daté du 21 juin contient 31 avis postérieurs, jusqu'au 29 juin.
Et les 22 annonces de la première règle viennent de 10 hôtes seulement, dont
quatre en portent 16 : pas une erreur dispersée au hasard, quelques comptes
dont l'inscription est postérieure à l'historique de leurs annonces.

### Constater n'est pas décider

Deux anomalies constatées et chiffrées. La tentation est de supprimer les
lignes fautives. Résistez-y : une anomalie appelle d'abord une **hypothèse**
sur son origine, et c'est l'hypothèse qui commande la décision.

| Anomalie | Combien | Hypothèse | Décision |
|---|---|---|---|
| premier avis avant l'inscription | 22 lignes, 10 hôtes | ? | ? |
| avis après la date du fichier | 31 lignes | ? | ? |


### Hack-Time 🛠️

Nous venons de chiffrer deux incohérences de dates. À votre tour de remplir
les deux colonnes vides du tableau, dans la cellule ci-dessous :

1. Pour les 22 annonces dont le premier avis précède l'inscription de l'hôte :
   **erreur ou réalité** ? Proposez une explication plausible. Une piste :
   que se passe-t-il quand une agence de gestion reprend des logements déjà
   en location, ou quand un hôte recrée son compte ?
2. Même question pour les 31 avis postérieurs au 21 juin. Une piste : un
   fichier de 6 721 annonces se collecte-t-il en une seconde ?
3. Pour chacune : supprimez-vous les lignes, corrigez-vous la date, ou les
   gardez-vous telles quelles ? Justifiez en une phrase.

**Quatre minutes.**

**Défi optionnel**
Au notebook 3, nous calculerons le temps écoulé entre le dernier avis et le
21 juin 2025, en années. Quelle valeur impossible les 31 avis tardifs vont-ils
produire ? Et qui s'en apercevra si personne ne l'a cherchée ?

*Vos réponses :*

1. Premier avis avant l'inscription :
2. Avis après la date du fichier :
3. Décisions :

## À retenir

- Explorez avant de nettoyer, toujours avec la même routine : `shape`,
  `info()`, `describe()`, `value_counts()`, un histogramme titré. Chaque outil
  a un angle mort.
- Un prix `$1,234.00`, un taux `100%`, un statut `t` : convertissez-les
  explicitement, et vérifiez le résultat.
- Une conversion ne doit jamais inventer d'information : `== "t"` transforme
  les manquants en « non », `map` les laisse manquants.
- Constater n'est pas décider : une anomalie appelle une hypothèse, et la
  décision s'écrit avec sa justification.

**La suite** : le notebook 2 s'attaque aux valeurs manquantes, aux doublons et
aux valeurs extrêmes, comme ce prix de 9 999 par nuit.
